(reproducibilidad-inferencia)=
# 12. Reproducibilidad y procedencia

El libro se reconstruye desde los datos del repositorio y conserva las salidas de cada análisis. El entorno documental se separa del entrenamiento. Las rutas se resuelven a partir de la raíz del repositorio y las fechas se interpretan en America/Bogota.

La {ref}`sección de inferencia en vivo <inferencia-en-vivo>` permite ejecutar los modelos en el navegador, modificar su historia y descargar las trayectorias.

In [1]:
from pathlib import Path
import sys,platform,json,hashlib,importlib.metadata as metadata
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/processed/panel_horario.parquet').exists())
paquetes=['jupyter-book','nbformat','ipython','matplotlib','pandas','pyarrow','scipy','statsmodels','scikit-learn']
entorno={'python':sys.version,'sistema':platform.platform(),
    'dependencias':{p:metadata.version(p) for p in paquetes},'dispositivo_eda':'CPU',
    'zona_horaria':'America/Bogota'}
(ROOT/'results/jbook/entorno.json').write_text(json.dumps(entorno,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.Series(entorno['dependencias'],name='versión documental'))


jupyter-book     1.0.4
nbformat        5.11.1
ipython         9.18.1
matplotlib      3.11.2
pandas           3.1.1
pyarrow         26.0.1
scipy           1.18.1
statsmodels     0.15.0
scikit-learn     1.9.1
Name: versión documental, dtype: str

## Manifiesto de entradas del análisis

Las huellas SHA-256 identifican el contenido binario exacto. Se incluyen fuentes numéricas, titulares, archivos externos y el panel consolidado; los resultados de modelos se identifican por separado mediante su registro de artefactos.

In [2]:
def sha256_archivo(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for bloque in iter(lambda:f.read(1024*1024),b''):
            h.update(bloque)
    return h.hexdigest()

rutas=set(p for p in (ROOT/'data/raw/numeric').rglob('*') if p.is_file())
rutas.update(p for p in (ROOT/'data/external').rglob('*') if p.is_file())
rutas.update([ROOT/'data/raw/noticias/titulares.parquet',ROOT/'data/processed/panel_horario.parquet',
    ROOT/'data/processed/feature_registry.json',ROOT/'data/processed/diccionario_datos.json'])
manifiesto=pd.DataFrame([{'archivo':p.relative_to(ROOT).as_posix(),'bytes':p.stat().st_size,
    'sha256':sha256_archivo(p)} for p in sorted(rutas)])
manifiesto.to_csv(ROOT/'results/jbook/manifiesto_entradas.csv',index=False)
n_archivos=len(manifiesto); total_gb=float(manifiesto['bytes'].sum()/1e9)


El manifiesto registra 59 archivos y 0.06 GB de entradas. Sus huellas permiten detectar cambios de versión antes de reconstruir el análisis. El diccionario y el registro de disponibilidad complementan la identidad binaria con las definiciones semánticas y temporales.

## Ejecución en Windows y Linux

Crear un entorno con la versión de Python registrada y usar el archivo de dependencias fijadas. Los comandos siguientes se ejecutan desde la raíz; la activación del entorno depende del sistema operativo.

### Windows

```powershell
python -m venv .bookenv
.bookenv\Scripts\python.exe -m pip install -r requirements-book.lock.txt
.bookenv\Scripts\jupyter-book.exe build book --warningiserror --keep-going
.bookenv\Scripts\python.exe scripts\verificar_jbook.py
```

### Linux

```bash
python3 -m venv .bookenv
.bookenv/bin/python -m pip install -r requirements-book.lock.txt
.bookenv/bin/jupyter-book build book --warningiserror --keep-going
.bookenv/bin/python scripts/verificar_jbook.py
```

Estos comandos construyen el libro con las salidas conservadas. `execute_notebooks: off` evita reevaluar las celdas durante esa construcción. Para recalcular los análisis se utiliza `python scripts/crear_jbook.py`; para reconstruir solo la evaluación, `python scripts/reorganizar_capitulo10.py`. La recalculación requiere las entradas e instantáneas indicadas en la procedencia; los diagnósticos de redes conservan su entorno separado.

Los análisis documentales se ejecutan por CPU. El entrenamiento de las configuraciones seleccionadas utiliza `requirements-entrenamiento.txt` y `scripts/entrenar_arquitectura.py`, con `--dispositivo auto`, `cpu` o `cuda`. La instalación de PyTorch debe ser compatible con el dispositivo y su controlador. El código completo y los comandos por modelo se muestran en el {ref}`pipeline compartido <entrenamiento-portable>`. La comprobación funcional de las 15 configuraciones se ejecutó en Windows con CPU; Linux y GPU requieren comprobación en sus propios entornos.

## Construcción y verificación del libro

El verificador comprueba estructura de cuadernos, celdas ejecutadas, ausencia de errores, interpretaciones junto a las figuras, cobertura del diccionario y enlaces HTML locales. Después copia el HTML construido a docs. Los archivos del libro y sus evidencias se conservan en book, docs y results/jbook.

## Procedencia de modelos y reconstrucción

Las cuatro familias de demanda/calendario conservan el commit d1ae9a4e09dd1dead4af1329f16aa3e8d0f2c3d3 y su [registro de artefactos](_static/modelos/modelos.json); el [índice SHA-256](_static/modelos/indice_artefactos_v3_v4.csv) identifica 2.304 archivos. Las arquitecturas ampliadas proceden del commit 66111003f387b8452b1fc690dbc7fc503d5570df y de los paquetes consolidados. El inventario conserva pesos, escaladores, historias, predicciones y métricas por semilla; el control N compartido se cuenta una vez. TFT utiliza el paquete seleccionado por rigor del procedimiento y fidelidad arquitectónica.

El benchmark de referencia integra 75 realizaciones finales de quince procedimientos, con cinco semillas cada uno, y dos referencias deterministas. M4ₙ añade diez realizaciones alineadas con los mismos orígenes. Las métricas se reconstruyen con predicciones conservadas y orígenes alineados. [Metadatos de ajuste y validación](_static/resultados/evaluacion_e2/metadatos_procedimientos.csv) y [cobertura de diagnósticos](_static/resultados/evaluacion_e2/cobertura_diagnosticos.csv) identifican el alcance de cada análisis. TGForecaster conserva cinco realizaciones por variante.

Para reconstruir la evaluación integrada se ejecuta `python scripts/reorganizar_capitulo10.py`; después, `jupyter-book build book --warningiserror --keep-going` y `python scripts/verificar_jbook.py`. El generador completo `scripts/crear_jbook.py` integra la misma ampliación al finalizar. Los datos de predicción y las entradas documentales deben conservar su inventario de procedencia. La reproducción del análisis documental usa CPU; la inferencia y el entrenamiento requieren PyTorch compatible con CPU, CUDA o ROCm según el procedimiento y dispositivo. La reconstrucción del libro no inicia entrenamiento.

## Reconstrucción y revisión de consistencia

La narrativa de resultados se genera mediante `scripts/narrativa_resultados.py` a partir del benchmark, contrastes, MCS, selección y diagnósticos conservados. Las celdas Markdown identifican su función mediante metadatos; las cifras y conclusiones se actualizan durante la reconstrucción.

La {download}`revisión vigente <_static/resultados/revision_documental/revision_consistencia.md>` y el [resumen con huellas de procedencia](_static/resultados/revision_documental/resumen_documental.json) documentan el inventario y las fuentes utilizadas.

## Reproducción de la evaluación y sus diagnósticos

Las fuentes de `book/` incluyen los cuadernos con sus salidas, las fichas de arquitectura, los esquemas y los archivos descargables. `jupyter-book build book --warningiserror --keep-going` construye el HTML desde esas fuentes, con la ejecución de cuadernos desactivada.

Los CSV y NPZ de `results/evaluacion_e2/` y `results/diagnosticos_ampliados/` contienen las pérdidas, contrastes, curvas y mediciones utilizadas en las figuras. La reconstrucción de la presentación utiliza estos archivos y las predicciones conservadas:

```bash
python scripts/reorganizar_capitulo10.py
jupyter-book build book --warningiserror --keep-going
python scripts/verificar_jbook.py
```

`scripts/actualizar_diagnosticos_jbook.py` actualiza las figuras a partir de las mismas mediciones. `scripts/reconstruir_evaluacion.py` vuelve a ejecutar el código analítico del cuaderno integrado con las entradas verificadas por SHA-256. Ninguno de estos procedimientos reajusta modelos.

La repetición de inferencia requiere, además, los checkpoints, escaladores y vectores E5 identificados en `fuentes_diagnosticos.json`. Su ubicación se configura mediante `DEMANDA_ARTEFACTOS`, apuntando al directorio que contiene `consolidado/indice_modelos.json` y `fuentes/`. Estos archivos binarios no son necesarios para construir el libro ni reconstruir las gráficas desde las mediciones. Las nuevas mediciones utilizan el entorno `requirements-inferencia.lock.txt` y los comandos siguientes:

```bash
python scripts/diagnosticos_ampliados.py --fase inferir --workers 4
python scripts/diagnosticos_ampliados.py --fase latencia
```

`reproduccion_semillas.csv` documenta la correspondencia con las predicciones conservadas; `entorno.json` identifica las condiciones de la medición de latencia.

## N-BEATS y TimesNet: entradas y reproducción

Las claves `nuevo_nbeats__N` y `nuevo_timesnet__N` identifican cinco realizaciones de cada red. Configuraciones, escaladores, historias y predicciones canónicas se conservan en `results/evaluacion_e2/entradas/adicionales/`. TimesNet utiliza `yhat_lote1_kwh` de la reevaluación por origen; N-BEATS conserva sus predicciones originales. Las etiquetas provienen del panel congelado en doble precisión.

La selección y la parada interna mantienen sus registros de desarrollo. Para TimesNet, la prueba y las perturbaciones reciben un único origen en cada llamada. Los pesos se verifican mediante el [inventario SHA-256 de diez checkpoints](_static/resultados/top10_adicionales/auditoria_pesos.csv); la [procedencia de la integración](_static/resultados/top10_adicionales/integracion.json) y el [protocolo de diagnósticos](_static/resultados/top10_adicionales/protocolo_diagnosticos.json) acompañan los resultados.

`integrar_top10.py` normaliza las entradas y mantiene la copia de procedencia. `estadistica_e2.py` calcula las familias completas de contrastes sobre las predicciones guardadas. `reorganizar_capitulo10.py` reconstruye la evaluación y las fichas de ambas arquitecturas; la aplicación estática utiliza los mismos resultados mediante su exportador.

```console
python scripts/estadistica_e2.py
python scripts/reorganizar_capitulo10.py
python dashboard/scripts/export_data.py
jupyter-book build book --warningiserror --keep-going
python scripts/verificar_jbook.py
```

Estos pasos leen los resultados disponibles. Los pipelines de ajuste e inferencia se documentan en las fichas de arquitectura y permanecen separados de la construcción del libro.

## Reproducción de M4ₙ

`scripts/integrar_m4_resultados.py` reconstruye las métricas, tablas y figuras de M4ₙ desde las predicciones e historiales de diez semillas conservadas en `results/m4_cierre_gpu_20261123/reducido_B/`. La escala semanal y los comparadores proceden del benchmark; TimesNet se verifica con sus cinco predicciones de lote uno. Los bloques analíticos se muestran completos en la sección de resultados.

Ejecutar `python scripts/integrar_m4_resultados.py` desde la raíz regenera únicamente la documentación y los resúmenes derivados. El manifiesto registra las huellas de los archivos utilizados. No se importan módulos de redes ni se cargan pesos; los contrastes se leen de los resultados guardados y conservan sus ajustes por multiplicidad.

(inferencia-portable)=
## Evaluación complementaria e inferencia portable

Los archivos de `results/cierre_e3_sin_entrenar_20261124/` conservan las pérdidas diarias, las 102 comparaciones complementarias, el MCS, los intervalos y los diagnósticos del núcleo y de eficiencia. `scripts/integrar_cierre_e3.py` reconstruye tablas y figuras desde estos resultados; los módulos de medición y exportación se muestran completos en la evaluación.

El [paquete CPU](_static/resultados/cierre_e3/M4_inferencia_CPU_portable.zip) contiene los tres expertos, M1 y la memoria en cinco módulos ONNX, escaladores, contrato de entradas preparadas, ejemplos y manifiesto. Se verificó en un entorno Windows nuevo con Python 3.13, NumPy y ONNX Runtime. Los datos propios deben respetar la alineación temporal y las transformaciones del contrato. El contenedor Linux queda preparado mediante Dockerfile y su ejecución requiere verificación en esa plataforma.

Después de extraer el paquete, su comprobación se ejecuta con:

```console
python -m pip install -r requirements.txt
python ejecutar.py --verificar
```

### Correspondencia métrica de la exportación

El panel prefijado de verificación contiene cuatro fechas históricas y 96 intervalos, con la semilla 101. El MASE nativo es 0,42716155 y el exportado 0,42716146; la diferencia es -9.36e-08. La comparación se limita al panel de verificación y no sustituye el benchmark de 1.461 días y diez realizaciones. [Resultados por fecha](_static/resultados/auditoria_e3/roundtrip_metricas.csv).

```{literalinclude} _static/codigo_e3/cierre/auditar_exportacion_score_e3.py
:language: python
```
